# 📍 K-Nearest Neighbors
**ISLP Chapters 2 & 4 · Pattern Recognition for the Rest of Us**

> The simplest ML model: to predict a new point, find its K closest neighbors and vote (classification) or average (regression). No training, no coefficients, no assumptions — just memory and distance.

### What you'll learn
- KNN classification and regression
- How K controls the bias-variance tradeoff
- The curse of dimensionality — why KNN struggles at high dimensions
- Choosing K with cross-validation
- Standardizing features before KNN

### Dataset: Iris (classification) + Boston Housing (regression)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, mean_squared_error

In [ ]:
iris = load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names
print("Iris dataset:", X_iris.shape)
print("Classes:", class_names)
print("\nFirst 3 rows:")
X_iris.head(3)

## 📐 Part 1 — How KNN Works

To classify a new point x₀:
1. Compute distance from x₀ to every training point
2. Find the K nearest neighbors
3. Predict the majority class among those K neighbors

For regression: predict the **mean** of the K neighbors' values.

The only hyperparameter is **K**:
- Small K (K=1) → very flexible, low bias, high variance, overfits
- Large K → smoother boundary, higher bias, lower variance, may underfit

In [ ]:
# Visualize KNN decision boundaries for different K values
from sklearn.inspection import DecisionBoundaryDisplay

# Use only 2 features for visualization
X_2feat = X_iris[['sepal length (cm)', 'petal length (cm)']].values
y = y_iris.values

scaler = StandardScaler()
X_s = scaler.fit_transform(X_2feat)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
K_vals = [1, 5, 15, 50]
colors = ['#1e5fa8','#e85d20','#1a7a45']

for ax, K in zip(axes, K_vals):
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X_s, y)
    
    xx, yy = np.meshgrid(np.linspace(-3,3,200), np.linspace(-3,3,200))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.25, colors=colors)
    ax.contour(xx, yy, Z, colors=['#1e5fa8','#e85d20','#1a7a45'], linewidths=1)
    for cls, color, name in zip([0,1,2], colors, class_names):
        mask = y==cls
        ax.scatter(X_s[mask,0], X_s[mask,1], color=color, s=20, alpha=0.7, label=name)
    
    cv_acc = cross_val_score(KNeighborsClassifier(n_neighbors=K), X_s, y, cv=10).mean()
    ax.set_title(f'K = {K}\n10-fold CV acc = {cv_acc:.3f}', fontsize=10)
    ax.set_xlabel('Sepal length (std)')
    if K==1: ax.set_ylabel('Petal length (std)')

plt.suptitle('KNN Decision Boundaries — Effect of K', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 K=1 memorizes training data perfectly — jagged boundary, overfits")
print("   K=50 smooths everything — may miss fine distinctions")

In [ ]:
# Choose K with cross-validation
K_range = range(1, 51)
cv_scores = []
for K in K_range:
    knn = KNeighborsClassifier(n_neighbors=K)
    scores = cross_val_score(knn, X_s, y, cv=10, scoring='accuracy')
    cv_scores.append(scores.mean())

best_K = K_range[cv_scores.index(max(cv_scores))]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(K_range), cv_scores, 'o-', color='#1e5fa8', lw=1.5, markersize=4)
ax.axvline(best_K, color='#e85d20', lw=2, ls='--', label=f'Best K={best_K} (acc={max(cv_scores):.3f})')
ax.set_xlabel('K (number of neighbors)')
ax.set_ylabel('10-Fold CV Accuracy')
ax.set_title('Choosing K via Cross-Validation — Iris dataset')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\n📌 Optimal K = {best_K}")
print("   K=1 often appears good on training data but CV reveals the true optimum")

In [ ]:
# Curse of dimensionality
# As dimensions increase, all points become equally distant from each other
dims = [2, 5, 10, 20, 50, 100, 200]
n_samples = 1000
ratios = []

np.random.seed(42)
for d in dims:
    X_d = np.random.uniform(0, 1, (n_samples, d))
    dists = np.linalg.norm(X_d - X_d[0], axis=1)[1:]
    ratio = dists.std() / dists.mean()  # lower = more uniform = worse for KNN
    ratios.append(ratio)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dims, ratios, 'o-', color='#e85d20', lw=2, markersize=7)
ax.set_xlabel('Number of Dimensions')
ax.set_ylabel('Std(distances) / Mean(distances)')
ax.set_title('Curse of Dimensionality — distances become uniform in high dimensions\n(KNN loses its meaning when all points are equally "near")')
plt.tight_layout()
plt.show()
print("\n📌 In 2D, neighbors really are nearby. In 200D, the 'nearest' neighbor is almost as far as any other point.")
print("   KNN works best with <20 features. Consider PCA first for high-dimensional data.")

In [ ]:
answers = {
    "q1": "",  # How does KNN make a prediction for a new point?
    "q2": "",  # What happens to bias and variance as K increases?
    "q3": "",  # Why must features be standardized before applying KNN?
    "q4": "",  # What is the curse of dimensionality and how does it affect KNN?
    "q5": "",  # KNN has no explicit training phase. What is the cost of this at prediction time?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
# ── Submit your results to the instructor ────────────────────────────────────
# Fill in your GitHub username (do this once, it stays saved in the notebook)
GITHUB_USERNAME = ""   # ← put your GitHub username here e.g. "jsmith42"

# ─────────────────────────────────────────────────────────────────────────────
# DO NOT EDIT BELOW THIS LINE
import json, urllib.parse

def build_submission_url(username, notebook_name, score, total, answers_dict,
                          form_base_url=None, entries=None):
    """Build a pre-filled Google Form URL for quiz submission."""
    if not form_base_url:
        print("⚠  No submission URL configured yet.")
        print("   Your instructor will share the Form URL — paste it into form_base_url below.")
        print("   Your answers are saved in this notebook regardless.")
        return None

    params = {
        entries.get('user',     'entry.000000001'): username,
        entries.get('notebook', 'entry.000000002'): notebook_name,
        entries.get('score',    'entry.000000003'): str(score),
        entries.get('total',    'entry.000000004'): str(total),
    }
    url = form_base_url.replace('/viewform','') + '/viewform?' + urllib.parse.urlencode(params)
    return url

# ── Submission config (your instructor fills these in) ───────────────────────
FORM_BASE_URL = None   # e.g. "https://docs.google.com/forms/d/e/XXXX/viewform"
FORM_ENTRIES  = {      # entry IDs from the Google Form
    'user':     'entry.000000001',
    'notebook': 'entry.000000002',
    'score':    'entry.000000003',
    'total':    'entry.000000004',
}
# ─────────────────────────────────────────────────────────────────────────────

# Calculate score from answers dict
score_val = sum(1 for v in answers.values() if str(v).strip())  # counts answered Qs
# For auto-graded notebooks the score is passed directly
# Here we just verify completion
n_answered = sum(1 for v in answers.values() if str(v).strip())
n_total    = len(answers)

print(f"\n{'='*50}")
print(f"Quiz completion: {n_answered}/{n_total} questions answered")
if n_answered < n_total:
    print(f"⚠  Please answer all {n_total} questions before submitting.")
else:
    print("✅ All questions answered!")
    if GITHUB_USERNAME:
        import os
        nb_name = os.path.basename(globals().get('__vsc_ipynb_file__',
                  globals().get('__file__', 'unknown-notebook'))).replace('.ipynb','')
        url = build_submission_url(GITHUB_USERNAME, nb_name,
                                   n_answered, n_total, answers,
                                   FORM_BASE_URL, FORM_ENTRIES)
        if url:
            print(f"\n🔗 Submit to instructor:")
            print(f"   {url}")
            print("\n   Copy the URL above, open it in a new tab, and click Submit.")
        else:
            print("\n   Save your notebook to GitHub to record your progress:")
            print("   File → Save a copy in GitHub → choose your fork")
    else:
        print("\n⚠  Add your GitHub username to GITHUB_USERNAME above, then re-run.")
    print(f"{'='*50}")

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*